# Income Qualification Analysis

## Project Description
This project aims to identify the level of income qualification needed for families in Latin America using a Proxy Means Test (PMT) approach. We'll analyze Costa Rican household characteristics to predict poverty levels.

## Objectives
1. Identify the output variable
2. Understand the type of data
3. Check for biases in the dataset
4. Ensure all family members have the same poverty level
5. Check for houses without family heads
6. Handle missing values
7. Build and evaluate a Random Forest classifier

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette('husl')

## 1. Load and Explore the Dataset

In [ ]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"\nTraining data columns: {len(train_df.columns)}")
print(f"Test data columns: {len(test_df.columns)}")

In [ ]:
train_df.head()

In [ ]:
train_df.info()

## 2. Identify the Output Variable

In [ ]:
print("Columns in training data:")
print(train_df.columns.tolist())
print("\nColumns in test data:")
print(test_df.columns.tolist())
print("\nColumns only in training data (likely target variable):")
train_only = set(train_df.columns) - set(test_df.columns)
print(train_only)

In [ ]:
if 'Target' in train_df.columns:
    target_col = 'Target'
elif len(train_only) == 1:
    target_col = list(train_only)[0]
else:
    target_col = None
    print("Target variable not clearly identified")

if target_col:
    print(f"Target variable identified: {target_col}")
    print(f"\nTarget variable distribution:")
    print(train_df[target_col].value_counts().sort_index())
    
    plt.figure(figsize=(8, 6))
    train_df[target_col].value_counts().sort_index().plot(kind='bar')
    plt.title(f'Distribution of {target_col}')
    plt.xlabel('Poverty Level')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

## 3. Understand the Type of Data

In [ ]:
print("Data types summary:")
print(train_df.dtypes.value_counts())
print("\nNumeric columns:")
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Count: {len(numeric_cols)}")
print("\nCategorical columns:")
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print(f"Count: {len(categorical_cols)}")
if categorical_cols:
    print(categorical_cols)

In [ ]:
train_df.describe()

## 4. Check for Biases in the Dataset

In [ ]:
if target_col:
    print("Checking for class imbalance:")
    target_dist = train_df[target_col].value_counts(normalize=True).sort_index()
    print(target_dist)
    
    plt.figure(figsize=(10, 6))
    target_dist.plot(kind='bar')
    plt.title('Target Variable Distribution (Normalized)')
    plt.xlabel('Poverty Level')
    plt.ylabel('Proportion')
    plt.xticks(rotation=0)
    plt.axhline(y=1/len(target_dist), color='red', linestyle='--', alpha=0.7, label='Balanced line')
    plt.legend()
    plt.show()
    
    max_class = target_dist.max()
    min_class = target_dist.min()
    imbalance_ratio = max_class / min_class
    print(f"\nImbalance ratio: {imbalance_ratio:.2f}")
    if imbalance_ratio > 3:
        print("⚠️  Dataset shows significant class imbalance")
    else:
        print("✅ Dataset appears reasonably balanced")

## 5. Check if All Members of the House Have the Same Poverty Level

In [ ]:
if 'idhogar' in train_df.columns and target_col:
    household_targets = train_df.groupby('idhogar')[target_col].nunique()
    inconsistent_households = household_targets[household_targets > 1]
    
    print(f"Total households: {len(household_targets)}")
    print(f"Households with inconsistent poverty levels: {len(inconsistent_households)}")
    print(f"Percentage of inconsistent households: {len(inconsistent_households)/len(household_targets)*100:.2f}%")
    
    if len(inconsistent_households) > 0:
        print("\n⚠️  Some households have members with different poverty levels")
        print("Sample of inconsistent households:")
        sample_inconsistent = inconsistent_households.head().index
        for household_id in sample_inconsistent:
            household_data = train_df[train_df['idhogar'] == household_id][['idhogar', target_col]]
            print(f"Household {household_id}:")
            print(household_data[target_col].value_counts())
            print()
    else:
        print("✅ All household members have consistent poverty levels")
else:
    print("Cannot check household consistency - missing 'idhogar' column or target variable")

## 6. Check if There is a House Without a Family Head

In [ ]:
if 'parentesco1' in train_df.columns and 'idhogar' in train_df.columns:
    households_with_head = train_df[train_df['parentesco1'] == 1]['idhogar'].nunique()
    total_households = train_df['idhogar'].nunique()
    households_without_head = total_households - households_with_head
    
    print(f"Total households: {total_households}")
    print(f"Households with family head: {households_with_head}")
    print(f"Households without family head: {households_without_head}")
    print(f"Percentage without head: {households_without_head/total_households*100:.2f}%")
    
    if households_without_head > 0:
        print("\n⚠️  Some households don't have a family head (parentesco1 = 1)")
        
        all_households = set(train_df['idhogar'].unique())
        households_with_head_set = set(train_df[train_df['parentesco1'] == 1]['idhogar'].unique())
        households_without_head_ids = all_households - households_with_head_set
        
        print(f"Sample households without head: {list(households_without_head_ids)[:5]}")
    else:
        print("✅ All households have a family head")
else:
    print("Cannot check for family heads - missing required columns")
    print("Available columns that might indicate family relationships:")
    relationship_cols = [col for col in train_df.columns if 'parent' in col.lower() or 'head' in col.lower() or 'rel' in col.lower()]
    print(relationship_cols)

## 7. Set Poverty Level of Members and Head Same in a Family

In [ ]:
if 'idhogar' in train_df.columns and 'parentesco1' in train_df.columns and target_col:
    print("Setting consistent poverty levels within families...")
    
    train_df_corrected = train_df.copy()
    
    head_targets = train_df_corrected[train_df_corrected['parentesco1'] == 1][['idhogar', target_col]]
    head_targets = head_targets.set_index('idhogar')[target_col].to_dict()
    
    for household_id, head_target in head_targets.items():
        train_df_corrected.loc[train_df_corrected['idhogar'] == household_id, target_col] = head_target
    
    household_targets_after = train_df_corrected.groupby('idhogar')[target_col].nunique()
    inconsistent_after = household_targets_after[household_targets_after > 1]
    
    print(f"Inconsistent households after correction: {len(inconsistent_after)}")
    
    if len(inconsistent_after) == 0:
        print("✅ All households now have consistent poverty levels")
        train_df = train_df_corrected
    else:
        print("⚠️  Some inconsistencies remain - may be households without heads")
else:
    print("Cannot set consistent poverty levels - missing required columns")

## 8. Count Null Values in Columns

In [ ]:
print("Null values count in training data:")
null_counts = train_df.isnull().sum()
null_counts_nonzero = null_counts[null_counts > 0].sort_values(ascending=False)

if len(null_counts_nonzero) > 0:
    print(f"\nColumns with null values: {len(null_counts_nonzero)}")
    print(null_counts_nonzero)
    
    plt.figure(figsize=(12, 6))
    null_counts_nonzero.plot(kind='bar')
    plt.title('Null Values Count by Column')
    plt.xlabel('Columns')
    plt.ylabel('Null Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    null_percentage = (null_counts_nonzero / len(train_df) * 100).round(2)
    print("\nNull percentages:")
    print(null_percentage)
else:
    print("✅ No null values found in the dataset")

print(f"\nTotal missing values: {train_df.isnull().sum().sum()}")
print(f"Dataset completeness: {(1 - train_df.isnull().sum().sum() / (train_df.shape[0] * train_df.shape[1])) * 100:.2f}%")

## 9. Remove Null Value Rows of Target Variable

In [ ]:
if target_col:
    initial_rows = len(train_df)
    target_nulls = train_df[target_col].isnull().sum()
    
    print(f"Initial number of rows: {initial_rows}")
    print(f"Rows with null target values: {target_nulls}")
    
    if target_nulls > 0:
        train_df_clean = train_df.dropna(subset=[target_col])
        final_rows = len(train_df_clean)
        
        print(f"Rows after removing null targets: {final_rows}")
        print(f"Rows removed: {initial_rows - final_rows}")
        print(f"Percentage removed: {((initial_rows - final_rows) / initial_rows) * 100:.2f}%")
        
        train_df = train_df_clean
        print("✅ Null target rows removed")
    else:
        print("✅ No null values in target variable")
else:
    print("Cannot remove null target rows - target variable not identified")

## 10. Prepare Data for Machine Learning

In [ ]:
if target_col:
    feature_cols = [col for col in train_df.columns if col != target_col]
    
    X = train_df[feature_cols]
    y = train_df[target_col]
    
    print(f"Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    
    remaining_nulls = X.isnull().sum().sum()
    if remaining_nulls > 0:
        print(f"\n⚠️  {remaining_nulls} null values remain in features")
        print("Filling null values with median for numeric and mode for categorical...")
        
        for col in X.columns:
            if X[col].dtype in ['object']:
                X[col].fillna(X[col].mode().iloc[0] if not X[col].mode().empty else 'Unknown', inplace=True)
            else:
                X[col].fillna(X[col].median(), inplace=True)
        
        print("✅ Null values filled")
    
    categorical_features = X.select_dtypes(include=['object']).columns
    if len(categorical_features) > 0:
        print(f"\nEncoding {len(categorical_features)} categorical features...")
        le_dict = {}
        
        for col in categorical_features:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            le_dict[col] = le
        
        print("✅ Categorical features encoded")
    
    print(f"\nFinal dataset shape: {X.shape}")
    print(f"Target classes: {sorted(y.unique())}")
else:
    print("Cannot prepare data - target variable not identified")

## 11. Predict Accuracy Using Random Forest Classifier

In [ ]:
if target_col and 'X' in locals():
    print("Training Random Forest Classifier...")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    rf_classifier = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1
    )
    
    rf_classifier.fit(X_train, y_train)
    
    y_pred = rf_classifier.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n🎯 Random Forest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix - Random Forest')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_classifier.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10))
    
    plt.figure(figsize=(10, 6))
    feature_importance.head(15).plot(x='feature', y='importance', kind='barh')
    plt.title('Top 15 Feature Importances - Random Forest')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Cannot train Random Forest - data not prepared")

## 12. Check Accuracy Using Random Forest with Cross-Validation

In [ ]:
if target_col and 'X' in locals():
    print("Performing Cross-Validation...")
    
    rf_cv = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1
    )
    
    cv_scores = cross_val_score(rf_cv, X, y, cv=5, scoring='accuracy', n_jobs=-1)
    
    print(f"\n📊 Cross-Validation Results (5-fold):")
    print(f"Individual fold scores: {[f'{score:.4f}' for score in cv_scores]}")
    print(f"Mean CV Accuracy: {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)")
    print(f"Standard Deviation: {cv_scores.std():.4f}")
    print(f"95% Confidence Interval: [{cv_scores.mean() - 2*cv_scores.std():.4f}, {cv_scores.mean() + 2*cv_scores.std():.4f}]")
    
    plt.figure(figsize=(8, 6))
    plt.boxplot(cv_scores)
    plt.title('Cross-Validation Accuracy Scores')
    plt.ylabel('Accuracy')
    plt.xlabel('Cross-Validation')
    plt.grid(True, alpha=0.3)
    
    plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: {cv_scores.mean():.4f}')
    plt.legend()
    plt.show()
    
    print(f"\n🎯 Final Model Performance Summary:")
    print(f"Single Train-Test Split Accuracy: {accuracy:.4f}")
    print(f"Cross-Validation Mean Accuracy: {cv_scores.mean():.4f}")
    
    if abs(accuracy - cv_scores.mean()) < 0.02:
        print("✅ Model performance is consistent between train-test and cross-validation")
    else:
        print("⚠️  Performance difference detected - model may be overfitting")
else:
    print("Cannot perform cross-validation - data not prepared")

## Summary and Conclusions

This analysis has completed all the required tasks for the Income Qualification project:

1. ✅ Identified the output variable
2. ✅ Understood the data types and structure
3. ✅ Checked for biases in the dataset
4. ✅ Verified household poverty level consistency
5. ✅ Checked for households without family heads
6. ✅ Ensured consistent poverty levels within families
7. ✅ Counted and handled null values
8. ✅ Removed null target variable rows
9. ✅ Built and evaluated a Random Forest classifier
10. ✅ Performed cross-validation analysis

The Random Forest model provides a robust approach to predicting income qualification levels for Latin American families using household characteristics, supporting the Proxy Means Test methodology.